In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Third Party
from datasets import load_dataset
from openai import OpenAI
from rich import print
from rich.panel import Panel
from sklearn.metrics import classification_report

# First Party
from sdg_hub import Flow, FlowMetadata, BlockRegistry

import nest_asyncio
nest_asyncio.apply()

/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Classifying news articles


In this tutorial, you’ll learn how to create your own custom data generation flow using SDG Hub. This notebook will walk you through all the essential pieces to make your own flow using `sdg_hub` for any use-case using the fundamental components of sdg_hub: `Blocks` and `Flows`

As an example use-case, we will pick news classification. Classification is a fundamental task in machine learning, where the goal is to assign predefined categories to input data. To address the classic machine learning use-case of news or text classification, we will use sdg_hub and leverage a language model to **classify news articles** with topic labels — specifically using the [AG News dataset](https://huggingface.co/datasets/fancyzhx/ag_news) from Hugging Face.

We’ll go step by step through a progressively improving flow. Each stage builds on the previous one, giving you a practical sense of how you can evolve your flow from using simple heuristics to highly customized and reliable data generation, using different inference paradigms such as self assessment.

### 🔍 Understand the Task
Before we write any prompts or code, we’ll take time to understand what we want the model to do. For this exercise, the task is **text classification** — assigning one of 4 possible categories (e.g., "World", "Sports", "Sci/Tech", "Business") to a given news article

### 🛠️ Build a Basic Annotation Flow and learn the `sdg_hub` way
We’ll start by creating a minimal flow that simply prompts the model to generate topic labels on the unlabeled data. This will use default prompts, simply populating the prompt with the text and asking the model to generate one of the 4 possible labels, with no examples.

### 🎯 Improve with Assessment and Iteration
Next, we’ll refine the flow by adding an assessment step. Iterations and self verification on a task often lead to better performance

Let’s get started by loading a sample of the dataset

In [3]:
dataset = load_dataset("fancyzhx/ag_news")

train_data = dataset["train"].shuffle(seed=42).select(range(500))
test_data = dataset["test"].shuffle(seed=42).select(range(100))

# map the labels to the category names
label_map = train_data.features['label'].names

train_data = train_data.map(lambda x: {"category": label_map[x["label"]]})
test_data = test_data.map(lambda x: {"category": label_map[x["label"]]})

In [4]:
# Group examples by category
examples_by_category = {}
for item in train_data:
    category = item['category']
    if category not in examples_by_category:
        examples_by_category[category] = []
    examples_by_category[category].append(item['text'])

# Print one example from each category in a panel
for category, examples in examples_by_category.items():
    print(Panel(examples[0], title=f"Category: {category}", expand=False))


╭──────────────────────────────────────────────── Category: World ────────────────────────────────────────────────╮
│ Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a      │
│ halt, the day after 18 people died in explosions at a political rally.                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Category: Sports ────────────────────────────────────────────────╮
│ Desiring Stability Redskins coach Joe Gibbs expects few major personnel changes in the offseason and wants to   │
│ instill a culture of stability in Washington.                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Category: Sci/Tech ───────────────────────────────────────────────╮
│ U2 pitches for Apple New iTunes ads airing during baseball games Tuesday will feature the advertising-shy Irish │
│ rockers.                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Category: Business ───────────────────────────────────────────────╮
│ Economy builds steam in KC Fed district The economy continued to strengthen in September and early October in   │
│ the Great Plains and Rocky Mountain regions covered by the Tenth Federal Reserve District, the Federal Reserve  │
│ Bank of Kansas City said Wednesday.                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Simple Data Annotation Pipeline

In this section, we’ll create our **first working flow** to perform classification using a language model. The goal is to understand the building blocks of `sdg_hub` and how we can employ them to get a language model to classify a given text.

### Recap: How `sdg_hub` Works

```mermaid
flowchart LR
    A[Flow] --> B[Blocks] --> C[Prompts]
    C --> D[Generated Data]
```

# Building a Simple Classification Flow

### Discover Blocks for us to use



In [5]:
BlockRegistry.discover_blocks()

                                                 Available Blocks                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Block Name                   ┃ Category   ┃ Description                                                         ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ CombineColumnsBlock          │ deprecated │ DEPRECATED: Use TextConcatBlock instead. Combines multiple columns  │
│                              │            │ into a single column using a separator                              │
│ DuplicateColumns             │ deprecated │ DEPRECATED: Use DuplicateColumnsBlock instead. Duplicates existing  │
│                              │            │ columns with new names according to a mapping dictionary            │
│ FilterByValueBlock           │ deprecated │ DEPRECATED: Use ColumnValueFilterBlock instead. Filters datasets    │
│                              │            │ based on column values using various comparison operations          │
│ FlattenColumnsBlock          │ deprecated │ DEPRECATED: Use MeltColumnsBlock instead. Transforms wide dataset   │
│                              │            │ format into long format by melting columns into rows                │
│ LLMBlock                     │ deprecated │ DEPRECATED: Use the new modular approach with PromptBuilderBlock,   │
│                              │            │ LLMChatBlock, and TextParserBlock instead                           │
│ RenameColumns                │ deprecated │ DEPRECATED: Use RenameColumnsBlock instead. Renames columns in a    │
│                              │            │ dataset according to a mapping dictionary                           │
│ SamplePopulatorBlock         │ deprecated │ DEPRECATED: Use a router block instead. Populates dataset with data │
│                              │            │ from configuration files                                            │
│ SelectorBlock                │ deprecated │ DEPRECATED: Use IndexBasedMapperBlock instead. Selects and maps     │
│                              │            │ values from one column to another                                   │
│ SetToMajorityValue           │ deprecated │ DEPRECATED: Use UniformColumnValueSetter with                       │
│                              │            │ reduction_strategy='mode' instead. Sets all values in a column to   │
│                              │            │ the most frequent value                                             │
│ EvaluateFaithfulnessBlock    │ evaluation │ Thin wrapper composing 4 blocks for faithfulness evaluation         │
│ EvaluateRelevancyBlock       │ evaluation │ Thin wrapper composing 4 blocks for relevancy evaluation            │
│ VerifyQuestionBlock          │ evaluation │ Thin wrapper composing 4 blocks for question verification           │
│ ColumnValueFilterBlock       │ filtering  │ Filters datasets based on column values using various comparison    │
│                              │            │ operations                                                          │
│ LLMChatBlock                 │ llm        │ Unified LLM chat block supporting 100+ providers via LiteLLM        │
│ LLMChatWithParsingRetryBlock │ llm        │ Composite block combining LLM chat and text parsing with automatic  │
│                              │            │ retry on parsing failures                                           │
│ LLMParserBlock               │ llm        │ Extracts specified fields from LLM response objects                 │
│ PromptBuilderBlock           │ llm        │ Formats prompts into structured chat messages or plain text using   │
│                              │            │ Jinja templates                                                     │
│ TextParserBlock              │ llm        │ Parses and

Summary: 25 blocks across 5 categories

It seems all the functionality we are interested in, such as building a prompt, chatting with an llm and parsing its output are under the `llm` category in sdg_hub. Lets start there.

In [6]:
from sdg_hub.core.blocks.llm import PromptBuilderBlock, LLMChatBlock, TextParserBlock, LLMParserBlock

### Creating the required blocks

To get started, we'll construct the simplest possible flow for text classification using SDG Hub. We will focus on 3 main blocks that will often appear as a triplet while using `sdg_hub`

1. **Prompt Builder Block**: Converts each input text into a prompt formatted for the LLM. The important input argument to keep in mind for  `PromptBuilderblock` is the `prompt_config_path` which is where the prompt template is saved. Any prompt engineering we would want to do would be done in such a prompt template.
2. **LLM Chat Block**: Sends the prompt to the language model and receives its response (the predicted label).
3. **Text Parser Block**: Extracts the final label from the LLM's output.

This setup results in a single LLM interaction per sample, forming a minimal classification pipeline.

We are going to be using the simple prompt that can be found in `news_articles_classification_prompt.yaml`

In [7]:
promptbuilderblock_1 = PromptBuilderBlock(block_name='annotation_prompt_builder', input_cols=['text'], output_cols=['annotation_prompt'], prompt_config_path="news_classification_prompt.yaml", format_as_messages=True)
llmchatblock_1 = LLMChatBlock(block_name='annotation_llm_chat_block', input_cols=['annotation_prompt'], output_cols=['raw_output'], temperature=0.0, max_tokens=5, extra_body={'guided_choice': ['World', 'Sports', 'Business', 'Sci/Tech']}, async_mode=True)
llmparserblock_1 = LLMParserBlock(block_name='annotation_llm_parser_block', input_cols=['raw_output'], extract_content=True, expand_lists=True)
textparserblock_1 = TextParserBlock(block_name='annotation_text_parser_block', input_cols=['annotation_llm_parser_block_content'], output_cols=['output'], start_tags=[''], end_tags=[''])

### Designing the `Flow`

The `Flow` class is at the heart of SDG Hub. Simply put, a `Flow` is a chain of `Blocks` that get executed sequentially. Here, we will simply chain our PromptBuilder -> LLMChatBlock -> TextParser, in that order:

```mermaid
flowchart LR
    subgraph Flow
        direction LR
        A[PromptBuilderBlock] --> B[LLMChatBlock] --> C[TextParserBlock]
    end
```



In [8]:
flow = Flow(blocks=[promptbuilderblock_1, llmchatblock_1, llmparserblock_1, textparserblock_1], metadata=FlowMetadata(name="annotation_flow", description="A flow for news article classification", author="sdg_hub"))

### Set the model configs for the `Flow`

In SDG Hub, model details such as the API base URL, the API Key (if any) and the model name are set at a Flow level using the `set_model_config` method as shown. The `model` parameter accepts a string in the format of "`provider`/`model_name`". Here our `provider` is 'hosted_vllm' as we are using a locally hosted model through vllm, and the model name is "meta-llama/Llama-3.3-70B-Instruct"

We must set the `api_base` parameter and point it to where the model endpoint can be found, in this case, `http://localhost:8000/v1`

In [9]:
# flow.set_model_config(model="hosted_vllm/meta-llama/Llama-3.3-70B-Instruct", api_base="http://localhost:8000/v1", api_key="")

flow.set_model_config(model="hosted_vllm/qwen3-8b", api_base="http://localhost:8101/v1", api_key="empty")


[10:06:17] INFO     Auto-detected 1 LLM blocks for configuration: ['annotation_llm_chat_block']         ]8;id=658891;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=320091;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#864\864]8;;\

[10:06:17] INFO     Loaded LLM client for model 'hosted_vllm/qwen3-8b'                         ]8;id=627415;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=276957;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

[10:06:17] INFO     Initialized LLMChatBlock 'annotation_llm_chat_block' with model           ]8;id=242169;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=904995;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#265\265]8;;\
                    'hosted_vllm/qwen3-8b'                                                                         

           INFO     Successfully configured 1 LLM blocks with: model: 'hosted_vllm/qwen3-8b', api_base: ]8;id=437987;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=440791;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#903\903]8;;\
                    'http://localhost:8101/v1', api_key: empty                                                     

           INFO     Configured blocks: ['annotation_llm_chat_block']                                    ]8;id=339542;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=314774;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#906\906]8;;\

### Time to generate!

In sdg_hub, the way to generate data is very simple. we simply use the `generate` method from `Flow`. At its simplest form, all the `generate` method needs is the input dataset to operate on. Additionally, we can pass runtime parameters for each block as well, if we wish to override any of the block specific model configs.

In [10]:
generated_data = flow.generate(test_data)

[10:06:18] INFO     Starting flow 'annotation_flow' v1.0.0 with 100 samples across 4 blocks             ]8;id=97893;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=886304;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#515\515]8;;\

           INFO     Executing block 1/4: annotation_prompt_builder (PromptBuilderBlock)                 ]8;id=404372;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=567188;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#670\670]8;;\

╭─────────────────────────────────────────── annotation_prompt_builder ───────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: PromptBuilderBlock                                                                                  │
│ Input Rows: 100                                                                                                 │
│ Input Columns: 3                                                                                                │
│ Column Names: text, label, category                                                                             │
│ Expected Output Columns: annotation_prompt                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── annotation_prompt_builder - Complete ──────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 100 → 100                                                                                                 │
│ Columns: 3 → 4                                                                                                  │
│ 🟢 Added: annotation_prompt                                                                                     │
│ 📋 Final Columns: annotation_prompt, category, label, text                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'annotation_prompt_builder' completed successfully: 100 samples, 4 columns    ]8;id=359395;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=302730;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

...

╭───────────────────────────────────────── annotation_text_parser_block ──────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: TextParserBlock                                                                                     │
│ Input Rows: 100                                                                                                 │
│ Input Columns: 6                                                                                                │
│ Column Names: text, label, category, annotation_prompt, raw_output, annotation_llm_parser_block_content         │
│ Expected Output Columns: output                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────── annotation_text_parser_block - Complete ────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 100 → 100                                                                                                 │
│ Columns: 6 → 7                                                                                                  │
│ 🟢 Added: output                                                                                                │
│ 📋 Final Columns: annotation_llm_parser_block_content, annotation_prompt, category, label, output, raw_output,  │
│ text                                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'annotation_text_parser_block' completed successfully: 100 samples, 7 columns ]8;id=267440;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=950506;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

╭────────────────────────────────────────── annotation_flow - Complete ───────────────────────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ annotation_prompt_b… │ PromptBuilderB… │      0.02s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_llm_chat… │ LLMChatBlock    │      1.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_llm_pars… │ LLMParserBlock  │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_text_par… │ TextParserBlock │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 4 blocks        │      1.05s │  100 final   │     7 final     │    4/4     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'annotation_flow' completed successfully: 100 final samples, 7 final columns   ]8;id=826255;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=498792;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#620\620]8;;\

### Evaluation

Now that we’ve generated synthetic labels using our simple classification flow, it’s time to evaluate how well the model performed. The goal of this section is to compare the predicted labels against the **true labels** from the dataset using standard classification metrics (precision, recall, f-1 score and classification accuracy)

We’ll use `sklearn.metrics.classification_report`, which provides precision, recall, F1-score, and support for each class.


In [11]:
print(classification_report(generated_data["category"], generated_data["output"]))

/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

precision    recall  f1-score   support

    Business       0.00      0.00      0.00        32
    Sci/Tech       0.00      0.00      0.00        19
      Sports       0.00      0.00      0.00        27
       World       0.22      1.00      0.36        22

    accuracy                           0.22       100
   macro avg       0.06      0.25      0.09       100
weighted avg       0.05      0.22      0.08       100

## Introducing an Assessment step

Our initial flow used a one step approach — the model was given the task, a fixed label set, and some input text. While this baseline gives us a useful starting point, it has clear limitations:

- The model may rely on generic heuristics or surface patterns that don’t generalize well.
- It can confuse similar categories (e.g., "World" vs. "Business") without knowing how they're typically used.
- Without guidance, the model may underperform on edge cases or ambiguous queries.


### What is Assessment

With an assessment step, we will call to the same LLM, but this time, we provide the LLM with its own previous categorization label, and the original text. We will prompt the LLM to think about the original prediction, and give it context about challening cases
In this manner, we can elicit critical judgement from the model about its own prior classification decision. This type of additional context can be useful in the next iteration.


### What We’ll Do Next

We’ll now enhance our flow by introducing another chain of `PromptBuilder` -> `LLMChatBlock` -> `TextParserBlock` whose purpose is to pass the (original text +  prediction) to the LLM and obtain a verification or assessment of the prediction.


```mermaid
flowchart LR
 subgraph Flow1[Initial Classification]
 direction LR
 A[PromptBuilderBlock] --> B[LLMChatBlock] --> C[TextParserBlock]
 end
 subgraph Flow2[Assessment]
 direction LR
 D[PromptBuilderBlock_Assessment] --> E[LLMChatBlock_Assessment] --> F[TextParserBlock_Assessment]
 end
 
 C --> D
```


We will investigate if this catches any of the mis-classifications, and get an idea of how well our verification prompting works!

In [12]:
promptbuilderblock_assessment = PromptBuilderBlock(block_name='verifier_prompt_builder', input_cols=['text', 'output'], output_cols=['assessment_prompt'], prompt_config_path="news_classification_assessment_prompt.yaml", format_as_messages=True)
llmchatblock_assessment = LLMChatBlock(block_name='verifier_llm_chat_block', input_cols=['assessment_prompt'], output_cols=['raw_assessment_output'], async_mode=True)
llmparserblock_assessment = LLMParserBlock(block_name='verifier_llm_parser_block', input_cols=['raw_assessment_output'], extract_content=True, expand_lists=True)
textparserblock_assessment = TextParserBlock(block_name='verifier_text_parser_block', input_cols=['verifier_llm_parser_block_content'], output_cols=['assessment_output'], start_tags=[''], end_tags=[''])

flow = Flow(blocks=[promptbuilderblock_1, llmchatblock_1, llmparserblock_1, textparserblock_1, promptbuilderblock_assessment, llmchatblock_assessment, llmparserblock_assessment, textparserblock_assessment], metadata=FlowMetadata(name="annotation_flow", description="A flow for news article classification", author="sdg_hub"))
# flow.set_model_config(model="hosted_vllm/meta-llama/Llama-3.3-70B-Instruct", api_base="http://localhost:8000/v1", api_key="")
flow.set_model_config(model="hosted_vllm/qwen3-8b", api_base="http://localhost:8101/v1", api_key="empty")



generated_data = flow.generate(test_data)

           INFO     Auto-detected 2 LLM blocks for configuration: ['annotation_llm_chat_block',         ]8;id=515784;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=536508;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#864\864]8;;\
                    'verifier_llm_chat_block']                                                                     

[10:06:19] INFO     Loaded LLM client for model 'hosted_vllm/qwen3-8b'                         ]8;id=248819;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=6989;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

           INFO     Initialized LLMChatBlock 'annotation_llm_chat_block' with model           ]8;id=458974;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=201618;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#265\265]8;;\
                    'hosted_vllm/qwen3-8b'                                                                         

           INFO     Loaded LLM client for model 'hosted_vllm/qwen3-8b'                         ]8;id=39548;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=124031;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

           INFO     Initialized LLMChatBlock 'verifier_llm_chat_block' with model             ]8;id=552338;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=399769;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#265\265]8;;\
                    'hosted_vllm/qwen3-8b'                                                                         

...

╭────────────────────────────────────────── verifier_text_parser_block ───────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: TextParserBlock                                                                                     │
│ Input Rows: 100                                                                                                 │
│ Input Columns: 10                                                                                               │
│ Column Names: text, label, category, annotation_prompt, raw_output, annotation_llm_parser_block_content,        │
│ output, assessment_prompt, raw_assessment_output, verifier_llm_parser_block_content                             │
│ Expected Output Columns: assessment_output                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── verifier_text_parser_block - Complete ─────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 100 → 100                                                                                                 │
│ Columns: 10 → 11                                                                                                │
│ 🟢 Added: assessment_output                                                                                     │
│ 📋 Final Columns: annotation_llm_parser_block_content, annotation_prompt, assessment_output, assessment_prompt, │
│ category, label, output, raw_assessment_output, raw_output, text, verifier_llm_parser_block_content             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'verifier_text_parser_block' completed successfully: 100 samples, 11 columns  ]8;id=725613;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=28148;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

╭────────────────────────────────────────── annotation_flow - Complete ───────────────────────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ annotation_prompt_b… │ PromptBuilderB… │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_llm_chat… │ LLMChatBlock    │      0.82s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_llm_pars… │ LLMParserBlock  │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_text_par… │ TextParserBlock │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_prompt_bui… │ PromptBuilderB… │      0.02s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_llm_chat_b… │ LLMChatBlock    │     31.11s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_llm_parser… │ LLMParserBlock  │      0.03s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_text_parse… │ TextParserBlock │      0.04s │  100 → 100   │       +1        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 8 blocks        │     32.05s │  100 final   │    11 final     │    8/8     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'annotation_flow' completed successfully: 100 final samples, 11 final columns  ]8;id=674008;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=438329;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#620\620]8;;\

In [13]:
generated_data_pd = generated_data.to_pandas()
mislabeled_samples = generated_data_pd[generated_data_pd["category"] != generated_data_pd["output"]]

print(Panel(mislabeled_samples.iloc[0]['assessment_output'], title="Assessment"))
print(Panel(str(mislabeled_samples.iloc[0]['category']), title="Ground truth label"))

╭────────────────────────────────────────────────── Assessment ───────────────────────────────────────────────────╮
│ <think>                                                                                                         │
│ Okay, let's tackle this classification problem. The text is about the Indian cricket board planning their own   │
│ telecast for an upcoming test series against Australia, which is part of a TV rights dispute. The predicted     │
│ category is World. I need to determine if that's correct or not.                                                │
│                                                                                                                 │
│ First, I'll recall the guidelines. The categories are World, Sports, Business, and Sci/Tech. Let me break down  │
│ the text. The key elements here are the Indian cricket board, broadcasting arrangements, a test series against  │
│ Australia, and a TV rights dispute.                                                                             │
│                                                                                                                 │
│ Looking at the categories, World news typically involves international affairs, politics, wars, or              │
│ country-specific events, including economic events of specific countries. Sports, on the other hand, covers     │
│ athletic competitions, teams, players, and sporting events. Business is about corporate earnings, market        │
│ movements, company performance, and economic indicators. Sci/Tech involves tech companies, innovations,         │
│ scientific discoveries, etc.                                                                                    │
│                                                                                                                 │
│ The text mentions the Indian cricket board, which is a sports organization. The event in question is a cricket  │
│ test series between India and Australia, which is a sporting event. However, there's a TV rights dispute here.  │
│ Now, TV rights disputes can sometimes be considered under Business if they involve corporate negotiations,      │
│ contracts, or financial aspects. But the main focus of the article is the cricket series itself and the board's │
│ decision to handle broadcasting, which is part of the sports event's logistics.                                 │
│                                                                                                                 │
│ Wait, but the dispute is about TV rights. That could be a business aspect because it's about contracts and      │
│ revenue. However, the primary context is the sports event. The Indian cricket board is a sports governing body, │
│ so their actions relate to sports management. The TV rights issue might be a business angle, but the article is │
│ primarily about the broadcasting arrangement for a cricket series, which is a sports event.                     │
│                                                                                                                 │
│ In the examples provided, there were cases where sports-related news was misclassified as Business or Sci/Tech. │
│ For instance, Example 3 was about Microsoft's acquisitions, which was classified as Sci/Tech. But in this case, │
│ the focus is on the cricket series and broadcasting, which are sports-related. However, the TV rights dispute   │
│ might be a business aspect. But the main subject is the sports event and the board's decision, not the business │
│ deal itself.                                                                                                    │
│                                                                                                                 │
│ Wait, the user's guidelines say that World includes country-specific events, including economic events of       │
│ specific countries. But the TV rights dispute is more 

╭────────────────────────────────────────────── Ground truth label ───────────────────────────────────────────────╮
│ Sports                                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Great! Now we can see that the assessment step is working good, especially on the misclassified samples as shown above. The above is a hard example which has slipped past our original classification flow, but was caught by our assessment step's critical judgement.

### Revising the Classifications

We will now create our final revision step, which will take the results of the initial prediction and the assessment steps and pass it onto the LLM once again for a revised attempt at classifying the same input text. The flow can be imagined like so:

```mermaid
flowchart LR
 subgraph Flow1[Initial Classification]
 direction LR
 A[PromptBuilderBlock] --> B[LLMChatBlock] --> C[TextParserBlock]
 end
 subgraph Flow2[Assessment]
 direction LR
 D[PromptBuilderBlock_Assessment] --> E[LLMChatBlock_Assessment] --> F[TextParserBlock_Assessment]
 end
 subgraph Flow3[Revised Classification]
 direction LR
 G[PromptBuilderBlock_Revision] --> H[LLMChatBlock_Revision] --> I[TextParserBlock_Revision]
 end
 
 C --> D
 F --> G
```

In [14]:
promptbuilderblock_revision = PromptBuilderBlock(block_name='revised_prompt_builder', input_cols=['text', 'output', 'assessment_output'], output_cols=['revised_prompt'], prompt_config_path="revise_news_classification_prompt.yaml", format_as_messages=True)
llmchatblock_revision = LLMChatBlock(block_name='revised_llm_chat_block', input_cols=['revised_prompt'], output_cols=['raw_revised_output'], temperature=0.0, max_tokens=5, extra_body={'guided_choice': ['World', 'Sports', 'Business', 'Sci/Tech']}, async_mode=True)
llmparserblock_revision = LLMParserBlock(block_name='revised_llm_parser_block', input_cols=['raw_revised_output'], extract_content=True, expand_lists=True)
textparserblock_revision = TextParserBlock(block_name='revised_text_parser_block', input_cols=['revised_llm_parser_block_content'], output_cols=['revised_output'], start_tags=[''], end_tags=[''])

flow = Flow(blocks=[promptbuilderblock_1, llmchatblock_1, llmparserblock_1, textparserblock_1, promptbuilderblock_assessment, llmchatblock_assessment, llmparserblock_assessment, textparserblock_assessment, promptbuilderblock_revision, llmchatblock_revision, llmparserblock_revision, textparserblock_revision], metadata=FlowMetadata(name="news_classification_flow", description="A flow for news article classification with assessment and revision", author="sdg_hub"))
# flow.set_model_config(model="hosted_vllm/meta-llama/Llama-3.3-70B-Instruct", api_base="http://localhost:8000/v1", api_key="")
flow.set_model_config(model="hosted_vllm/qwen3-8b", api_base="http://localhost:8101/v1", api_key="empty")

           INFO     Auto-detected 3 LLM blocks for configuration: ['annotation_llm_chat_block',         ]8;id=743582;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=204998;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#864\864]8;;\
                    'revised_llm_chat_block', 'verifier_llm_chat_block']                                           

[10:06:51] INFO     Loaded LLM client for model 'hosted_vllm/qwen3-8b'                         ]8;id=155038;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=248769;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

           INFO     Initialized LLMChatBlock 'annotation_llm_chat_block' with model           ]8;id=262126;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=32857;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#265\265]8;;\
                    'hosted_vllm/qwen3-8b'                                                                         

           INFO     Loaded LLM client for model 'hosted_vllm/qwen3-8b'                         ]8;id=599067;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=327654;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

           INFO     Initialized LLMChatBlock 'verifier_llm_chat_block' with model             ]8;id=683518;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=628513;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#265\265]8;;\
                    'hosted_vllm/qwen3-8b'                                                                         

           INFO     Loaded LLM client for model 'hosted_vllm/qwen3-8b'                         ]8;id=41332;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=324914;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

           INFO     Initialized LLMChatBlock 'revised_llm_chat_block' with model              ]8;id=825563;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=90764;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/llm_chat_block.py#265\265]8;;\
                    'hosted_vllm/qwen3-8b'                                                                         

           INFO     Successfully configured 3 LLM blocks with: model: 'hosted_vllm/qwen3-8b', api_base: ]8;id=922139;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=205951;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#903\903]8;;\
                    'http://localhost:8101/v1', api_key: empty                                                     

           INFO     Configured blocks: ['annotation_llm_chat_block', 'revised_llm_chat_block',          ]8;id=480737;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=52218;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#906\906]8;;\
                    'verifier_llm_chat_block']                                                                     

In [15]:
generated_data = flow.generate(test_data)

           INFO     Starting flow 'news_classification_flow' v1.0.0 with 100 samples across 12 blocks   ]8;id=118702;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=774724;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#515\515]8;;\

           INFO     Executing block 1/12: annotation_prompt_builder (PromptBuilderBlock)                ]8;id=285909;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=704995;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#670\670]8;;\

╭─────────────────────────────────────────── annotation_prompt_builder ───────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: PromptBuilderBlock                                                                                  │
│ Input Rows: 100                                                                                                 │
│ Input Columns: 3                                                                                                │
│ Column Names: text, label, category                                                                             │
│ Expected Output Columns: annotation_prompt                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── annotation_prompt_builder - Complete ──────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 100 → 100                                                                                                 │
│ Columns: 3 → 4                                                                                                  │
│ 🟢 Added: annotation_prompt                                                                                     │
│ 📋 Final Columns: annotation_prompt, category, label, text                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'annotation_prompt_builder' completed successfully: 100 samples, 4 columns    ]8;id=468308;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=321191;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

...

╭─────────────────────────────────────────── revised_text_parser_block ───────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: TextParserBlock                                                                                     │
│ Input Rows: 100                                                                                                 │
│ Input Columns: 14                                                                                               │
│ Column Names: text, label, category, annotation_prompt, raw_output, annotation_llm_parser_block_content,        │
│ output, assessment_prompt, raw_assessment_output, verifier_llm_parser_block_content, assessment_output,         │
│ revised_prompt, raw_revised_output, revised_llm_parser_block_content                                            │
│ Expected Output Columns: revised_output                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── revised_text_parser_block - Complete ──────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 100 → 100                                                                                                 │
│ Columns: 14 → 15                                                                                                │
│ 🟢 Added: revised_output                                                                                        │
│ 📋 Final Columns: annotation_llm_parser_block_content, annotation_prompt, assessment_output, assessment_prompt, │
│ category, label, output, raw_assessment_output, raw_output, raw_revised_output,                                 │
│ revised_llm_parser_block_content, revised_output, revised_prompt, text, verifier_llm_parser_block_content       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'revised_text_parser_block' completed successfully: 100 samples, 15 columns   ]8;id=835097;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=287671;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

╭────────────────────────────────────── news_classification_flow - Complete ──────────────────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ annotation_prompt_b… │ PromptBuilderB… │      0.00s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_llm_chat… │ LLMChatBlock    │      0.81s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_llm_pars… │ LLMParserBlock  │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ annotation_text_par… │ TextParserBlock │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_prompt_bui… │ PromptBuilderB… │      0.01s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_llm_chat_b… │ LLMChatBlock    │     25.92s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_llm_parser… │ LLMParserBlock  │      0.02s │  100 → 100   │       +1        │     ✓      │           │
│ │ verifier_text_parse… │ TextParserBlock │      0.03s │  100 → 100   │       +1        │     ✓      │           │
│ │ revised_prompt_buil… │ PromptBuilderB… │      0.02s │  100 → 100   │       +1        │     ✓      │           │
│ │ revised_llm_chat_bl… │ LLMChatBlock    │      2.12s │  100 → 100   │       +1        │     ✓      │           │
│ │ revised_llm_parser_… │ LLMParserBlock  │      0.03s │  100 → 100   │       +1        │     ✓      │           │
│ │ revised_text_parser… │ TextParserBlock │      0.02s │  100 → 100   │       +1        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 12 blocks       │     29.01s │  100 final   │    15 final     │   12/12    │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'news_classification_flow' completed successfully: 100 final samples, 15 final ]8;id=240812;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=477907;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#620\620]8;;\
                    columns                                                                                        

In [16]:
print(classification_report(generated_data["category"], generated_data["revised_output"]))

/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

precision    recall  f1-score   support

    Business       0.71      0.16      0.26        32
    Sci/Tech       0.67      0.11      0.18        19
      Sports       0.00      0.00      0.00        27
       World       0.24      1.00      0.39        22

    accuracy                           0.29       100
   macro avg       0.41      0.32      0.21       100
weighted avg       0.41      0.29      0.20       100

🔥 We improved the results drastically! Let us take a look at the number of mislabeled samples before and after the assessment + revision steps


In [17]:
generated_data_pd = generated_data.to_pandas()
num_mislabeled_output = (generated_data_pd["category"] != generated_data_pd["output"]).sum()
num_mislabeled_revised = (generated_data_pd["category"] != generated_data_pd["revised_output"]).sum()
print(f"Number of mislabeled samples (original output): {num_mislabeled_output}")
print(f"Number of mislabeled samples (revised output): {num_mislabeled_revised}")


Number of mislabeled samples (original output): 78

Number of mislabeled samples (revised output): 71

Great, we whave now improved the classification accuracy of our system by augmenting our naive classification flow by adding an assessment followed by a revision step


### Export the flow to yaml form


In [18]:
flow.to_yaml("news_classification_flow.yaml")

## ✅ Summary: What You’ve Learned

In this tutorial, you learned how to create your own flow for a custom use-case using `sdg_hub`, using the fundamental components: `Flow` and `Block`. You also learned how to create and structure the prompts. You learned how to design an assessment or a judgement step in order to improve the performance of the overall system. You started from scratch and evolved it into a robust, high-accuracy system.

## 🚀 What’s Next?

* Prompt Engineer! -  You can add examples for classifications directly in the classification steps and see how this improves the performance. In-context examples are extremely effective at aligning the model's outputs to the task at hand
* Try it out on your own data!